# MySQL Database

`MySQLConnection` reads CIM models from a MySQL database built with the ANSI SQL schema generated by [UCAiug CIMTool](https://cimtool.ucaiug.io). The connection is read-only.

The schema conventions it relies on:

* Each CIM class is its own table. A subclass row shares the `"mRID"` of its parent row (a Terminal has rows in `IdentifiedObject`, `ACDCTerminal` and `Terminal`). The class hierarchy is read from the `mRID -> mRID` foreign keys in `information_schema`.
* Associations are columns named after the role (e.g. `"Terminal"."ConductingEquipment"`) holding the target's mRID. Only the single-valued end is stored, so list ends such as `ConductingEquipment.Terminals` are rebuilt by reverse lookup using the data profile.
* Enumerations are stored as literal values (e.g. `'ABC'`) and converted to the data profile enum.
* Booleans are stored as 0/1 and numbers in SI units.
* Identifiers are double-quoted, so each session enables `ANSI_QUOTES`.

## Installation and Setup

`MySQLConnection` needs `mysql-connector-python`, from the `mysql` extra:

```bash
pip install "cim-graph[mysql]"
```

The repository's `docker-compose.yml` does not include MySQL. The examples need a running MySQL server with a CIMTool schema and a model loaded (here the IEEE 13-bus feeder in the `rc4_2021` profile, in the database `tier5_ieee13`), so the outputs are not shown on this page.

## Connection Parameters

The connection is configured with environment variables, read when it is created (see [Environment Variables](3_2_env_variables.ipynb)). `CIMG_USERNAME` and `CIMG_PASSWORD` have no defaults.

The queries match mRIDs as text, written in the `CIMG_IEC61970_552` style. This database stores them with a leading underscore (`_49AD8E07-...`), so it needs `552-LEGACY`; use `552-NEW` for a database whose mRIDs are bare UUIDs.

In [ ]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'rc4_2021'
os.environ['CIMG_IEC61970_552'] = '552-LEGACY'
os.environ['CIMG_HOST'] = '127.0.0.1'
os.environ['CIMG_PORT'] = '3306'
os.environ['CIMG_USERNAME'] = 'root'
os.environ['CIMG_PASSWORD'] = 'password'
os.environ['CIMG_DATABASE'] = 'tier5_ieee13'
os.environ['CIMG_USE_UNITS'] = 'true'

In [ ]:
from cimgraph.databases import MySQLConnection
from cimgraph.models import FeederModel
from cimgraph import utils

database = MySQLConnection()
cim = database.cim

## Building a Feeder Model

`get_object` takes the mRID as it is stored in the database and finds its concrete class by searching the class tables. `FeederModel` then loads all equipment in the feeder with their terminals and connectivity nodes.

In [ ]:
feeder = database.get_object('_49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(connection=database, container=feeder)

## Expanding the Graph

`get_all_edges` reads the class tables of each object, adds the associated objects and rebuilds the list ends by reverse lookup. With `CIMG_USE_UNITS=true`, unit values are `CIMUnit` objects in SI base units; convert them with `.to()`:

In [ ]:
network.get_all_edges(cim.ACLineSegment)
network.pprint(cim.ACLineSegment)

In [ ]:
line = network.first(cim.ACLineSegment)
print(f"Line {line.name}: length {line.length.to('m')} m ({line.length.to('ft'):.0f} ft)")

In [ ]:
# Expand the graph to all objects reachable from the feeder
utils.get_all_data(network)

In [ ]:
database.disconnect()